In [106]:
import pandas as pd
from pathlib import Path
import sys
import numpy as np
community_df = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\Community_matrix.csv")
processed_dir = Path(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output")

for csv_path in processed_dir.glob("*.csv"):
    globals()[csv_path.stem] = pd.read_csv(csv_path)

data_electorate = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\Electoral_Modelling\Electorate_Data.csv")
sub_region_map = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\sub_region_map.csv")


### Ward-level Electorate

Aggregate `Electorate_Demographics.csv` up from polygon (`Shape_ID`) to ward, using `df_polygon`'s `(District, District_Ward)` to assign each polygon to a ward, and `df_districts` as the full ward list (so every ward appears as a row even if it has no matching polygons).

In [107]:
demographic_cols = ["S", "A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2", "E1", "E2", "F1", "F2"]
df_electorates = Electorate_Demographics

In [ ]:
demographic_cols = ["S", "A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2", "E1", "E2", "F1", "F2"]

ward_base = df_districts[["District", "District_ID", "District_Ward", "Constituency"]].drop_duplicates(
    subset=["District", "District_Ward"]
)
ward_base = ward_base.merge(sub_region_map[["Constituency", "Sub_Region"]], on="Constituency", how="left")

polygon_ward_lookup = df_polygon[["Shape_ID", "District_Ward"]]
electorate_ward = Electorate_Demographics.merge(polygon_ward_lookup, on="Shape_ID", how="left")

ward_sum = electorate_ward.groupby(["District", "District_Ward"])[demographic_cols].sum().reset_index()

ward_electorate = ward_base.merge(ward_sum, on=["District", "District_Ward"], how="left")
ward_electorate[demographic_cols] = ward_electorate[demographic_cols].fillna(0).astype(int)

ward_electorate.to_csv(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\testWard_Electorate_Demographics.csv", index=False)

: 

### Link Electorate -> Turnout/Spoil -> First Round Results

Chain (per demographic cluster, per Shape_ID):

`Electors --x Turnout%--> Turnout_Votes --x (1 - Spoil%)--> Valid_Votes --x L%/MR%/R%--> Votes_L/MR/R`

`Sub_Region` is the common key across `df_electorates`, `Turnout_Spoil_Data.csv` and `Election_Result_first_round.csv`.
`R% = 100 - L% - MR%` since only Left and Middle-Right shares are given.

In [39]:
turnout_spoil = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\Electoral_Modelling\Turnout_Spoil_Data.csv")
election_result = pd.read_csv(r"E:\Coding Site\just_for_fun\Info_CSV\Electoral_Modelling\Election_Result_first_round.csv")

def wide_to_long(df, id_cols, value_name):
    return df.melt(id_vars=id_cols, value_vars=demographic_cols, var_name="Cluster", value_name=value_name)

turnout_rate = wide_to_long(
    turnout_spoil[turnout_spoil["Type"] == "Turnout"], ["Sub_Region"], "Turnout_Rate"
)
spoil_rate = wide_to_long(
    turnout_spoil[turnout_spoil["Type"] == "Spoil_General"], ["Sub_Region"], "Spoil_Rate"
)

general_results = election_result[election_result["Type"] == "First_Round_General"]
party_share = wide_to_long(general_results, ["Sub_Region", "Variable_1"], "Share")
party_share = (
    party_share.pivot_table(index=["Sub_Region", "Cluster"], columns="Variable_1", values="Share")
    .reset_index()
    .rename(columns={"L": "L_share", "MR": "MR_share"})
)
party_share["R_share"] = 100 - party_share["L_share"] - party_share["MR_share"]


In [40]:
electors_long = df_electorates.melt(
    id_vars=["County", "District", "Shape_ID", "Constituency", "Sub_Region"],
    value_vars=demographic_cols,
    var_name="Cluster",
    value_name="Electors",
)

electors_long = electors_long.merge(turnout_rate, on=["Sub_Region", "Cluster"], how="left")
electors_long = electors_long.merge(spoil_rate, on=["Sub_Region", "Cluster"], how="left")
electors_long = electors_long.merge(party_share, on=["Sub_Region", "Cluster"], how="left")

# round up to whole votes at each stage before feeding into the next
electors_long["Turnout_Votes"] = np.ceil(
    electors_long["Electors"] * electors_long["Turnout_Rate"] / 100
).astype(int)
electors_long["Valid_Votes"] = np.ceil(
    electors_long["Turnout_Votes"] * (1 - electors_long["Spoil_Rate"] / 100)
).astype(int)

for party, share_col in [("L", "L_share"), ("MR", "MR_share"), ("R", "R_share")]:
    electors_long[f"Votes_{party}"] = np.ceil(
        electors_long["Valid_Votes"] * electors_long[share_col] / 100
    ).astype(int)

electors_long.head()


,County,District,Shape_ID,Constituency,Sub_Region,Cluster,Electors,Turnout_Rate,Spoil_Rate,L_share,MR_share,R_share,Turnout_Votes,Valid_Votes,Votes_L,Votes_MR,Votes_R
0,A01,Alington City,A01_001,The City of Alington,Capital East,S,0,86.0,0.2,80.0,17.0,3.0,0,0,0,0,0
1,A01,Alington City,A01_002,The City of Alington,Capital East,S,308,86.0,0.2,80.0,17.0,3.0,265,265,212,46,8
2,A01,Alington City,A01_003,The City of Alington,Capital East,S,0,86.0,0.2,80.0,17.0,3.0,0,0,0,0,0
3,A01,Alington City,A01_004,The City of Alington,Capital East,S,126,86.0,0.2,80.0,17.0,3.0,109,109,88,19,4
4,A01,Alington City,A01_005,The City of Alington,Capital East,S,0,86.0,0.2,80.0,17.0,3.0,0,0,0,0,0


In [36]:
# totals & vote share by Sub_Region (the "regions" from Election_Result_first_round.csv)
vote_cols = ["Votes_L", "Votes_MR", "Votes_R"]

region_results = electors_long.groupby("Sub_Region")[vote_cols + ["Valid_Votes", "Electors"]].sum()
region_results = region_results.rename(columns={"Electors": "Total_Electorates"})

total_row = region_results.sum().rename("Total")
region_results = pd.concat([region_results, total_row.to_frame().T])

for party in ["L", "MR", "R"]:
    region_results[f"{party}_pct"] = region_results[f"Votes_{party}"] / region_results["Valid_Votes"] * 100

#region_results.to_csv(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\draftedRegion_Results.csv", index=True)

region_results_display = region_results.copy()
pct_cols = ["L_pct", "MR_pct", "R_pct"]
region_results_display[pct_cols] = region_results_display[pct_cols].map(lambda x: f"{x:.2f}%")

region_results_display

# same breakdown at Constituency level:
# constituency_results = electors_long.groupby(["Sub_Region", "Constituency"])[vote_cols + ["Valid_Votes"]].sum()


,Votes_L,Votes_MR,Votes_R,Valid_Votes,Total_Electorates,L_pct,MR_pct,R_pct
Capital East,5279762,3421347,1303236,9847080,11795672,53.62%,34.74%,13.23%
Capital West,1043287,1273070,725514,3012489,3819739,34.63%,42.26%,24.08%
Highland,2253643,3643338,1428911,7261837,8639323,31.03%,50.17%,19.68%
Midland,1679659,1924268,1572350,5078975,6355730,33.07%,37.89%,30.96%
Southeast,371815,379833,630773,1365398,1788859,27.23%,27.82%,46.20%
Total,10628166,10641856,5660784,26565779,32399323,40.01%,40.06%,21.31%


#### Ward-level Projected Vote Share & `election_status`

Aggregate `electors_long` (Votes_L/MR/R, Valid_Votes) from polygon to ward via `polygon_ward_lookup`/`ward_base` (from the Ward-level Electorate section above), then compute each party's vote share per ward.

`election_status` logic, ranking parties by vote share (winner/second/third) per ward:
- **Head to Head**: winner doesn't lead the runner-up by >=20 points (which also means it can't lead everyone by 20, since third's share <= second's).
- Otherwise there's a clear winner. Pick the challenger to report against it:
  - If second beats third by >15 points, challenger = second → `"Race: {winner}_{second}"`.
  - Otherwise second/third are themselves close, so the challenger is picked at random, weighted by their raw vote counts (`P(second) = second_votes / (second_votes + third_votes)`).

In [ ]:
ward_votes_sum = (
    electors_long.merge(polygon_ward_lookup, on="Shape_ID", how="left")
    .groupby(["District", "District_Ward"])[["Votes_L", "Votes_MR", "Votes_R", "Valid_Votes"]]
    .sum()
    .reset_index()
)

ward_vote_cols = ["Votes_L", "Votes_MR", "Votes_R", "Valid_Votes"]
ward_results = ward_base.merge(ward_votes_sum, on=["District", "District_Ward"], how="left")
ward_results[ward_vote_cols] = ward_results[ward_vote_cols].fillna(0)

for party in ["L", "MR", "R"]:
    ward_results[f"{party}_pct"] = np.where(
        ward_results["Valid_Votes"] > 0,
        ward_results[f"Votes_{party}"] / ward_results["Valid_Votes"] * 100,
        0.0,
    )

PARTIES = ["L", "MR", "R"]


def classify_ward(row):
    pct = {p: row[f"{p}_pct"] for p in PARTIES}
    votes = {p: row[f"Votes_{p}"] for p in PARTIES}
    winner, second, third = sorted(PARTIES, key=lambda p: pct[p], reverse=True)

    if pct[winner] - pct[second] < 15:
        return "Head to Head"

    if pct[second] - pct[third] > 15:
        challenger = second
    else:
        total = votes[second] + votes[third]
        prob_second = votes[second] / total if total > 0 else 0.5
        challenger = second if np.random.rand() < prob_second else third

    return f"Race: {winner}_{challenger}"


ward_results["election_status"] = ward_results.apply(classify_ward, axis=1)

ward_results.to_csv(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\draftedWard_Results.csv", index=False)

### Campaign Coverage per Polygon

Compare each polygon's centroid (`df_polygon.centroid`) against every canvass/service activity dot in `campaign.json` (`{Spectrum}_{Canvass|Service}` point sets, coordinates in the same map-unit space, `8.01` units = `1 km`).

Rules:
- **Canvass** radii: 0.5 / 1 / 1.5 km. **Service** radii: 1 / 2 / 3 km.
- An activity within multiple nested radii only counts once, at the **nearest** radius tier (e.g. 0.98km from a service point counts under the 1km tier only, not 2km/3km too).

Output: one row per polygon (`Shape_ID`), 18 columns `{L|MR|R}_{Canvass|Service}_{radius}` = count of that spectrum/type's activities whose nearest tier match is that radius.

In [42]:
import json
import ast

UNITS_PER_KM = 8.01

with open(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\campaign.json") as f:
    campaign = json.load(f)

RADII_KM = {
    "Canvass": [0.5, 1, 1.5],
    "Service": [1, 2, 3],
}
SPECTRUM_ABBR = {"Left": "L", "MR": "MR", "Right": "R"}


def fmt_km(r):
    return str(int(r)) if float(r).is_integer() else str(r)


poly_xy = np.array(df_polygon["centroid"].apply(ast.literal_eval).tolist())

coverage = pd.DataFrame({"Shape_ID": df_polygon["Shape_ID"]})

for spectrum, abbr in SPECTRUM_ABBR.items():
    for camp_type, radii in RADII_KM.items():
        cols = [f"{abbr}_{camp_type}_{fmt_km(r)}" for r in radii]
        for c in cols:
            coverage[c] = 0

        points = campaign.get(f"{spectrum}_{camp_type}", [])
        if not points:
            continue

        pts_xy = np.array([[p["x"], p["y"]] for p in points])

        diff = poly_xy[:, None, :] - pts_xy[None, :, :]
        dist_km = np.sqrt((diff ** 2).sum(axis=2)) / UNITS_PER_KM

        # nearest-tier-only assignment: each activity counts once, at the smallest radius it fits
        assigned_tier = np.full(dist_km.shape, -1)
        for i, r in enumerate(radii):
            unassigned = assigned_tier == -1
            assigned_tier[unassigned & (dist_km <= r)] = i

        for i, col in enumerate(cols):
            coverage[col] = (assigned_tier == i).sum(axis=1)

coverage.to_csv(
    r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\Campaign_Coverage.csv", index=False
)

coverage

,Shape_ID,L_Canvass_0.5,L_Canvass_1,L_Canvass_1.5,L_Service_1,L_Service_2,L_Service_3,MR_Canvass_0.5,MR_Canvass_1,MR_Canvass_1.5,MR_Service_1,MR_Service_2,MR_Service_3,R_Canvass_0.5,R_Canvass_1,R_Canvass_1.5,R_Service_1,R_Service_2,R_Service_3
0,A01_001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,A01_002,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,A01_003,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,A01_004,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,A01_005,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20845,WA06_001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
20846,WA07_001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
20847,WA08_001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
20848,WA09_001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Campaign Score per Polygon

Weight each tier and sum per spectrum: nearer tiers score higher.

| Tier | Weight |
|---|---|
| Canvass_0.5 | 3.5 |
| Canvass_1 | 2 |
| Canvass_1.5 | 1 |
| Service_1 | 3 |
| Service_2 | 2 |
| Service_3 | 1.5 |

In [43]:
TIER_WEIGHTS = {
    "Canvass_0.5": 3.5,
    "Canvass_1": 2,
    "Canvass_1.5": 1,
    "Service_1": 3,
    "Service_2": 2,
    "Service_3": 1.5,
}

score_table = pd.DataFrame({"Shape_ID": coverage["Shape_ID"]})
for abbr in SPECTRUM_ABBR.values():
    score_table[f"{abbr}_Score"] = sum(
        coverage[f"{abbr}_{tier}"] * weight for tier, weight in TIER_WEIGHTS.items()
    )

score_table

,Shape_ID,L_Score,MR_Score,R_Score
0,A01_001,0.0,0.0,0.0
1,A01_002,0.0,0.0,0.0
2,A01_003,0.0,0.0,0.0
3,A01_004,0.0,0.0,0.0
4,A01_005,0.0,0.0,0.0
...,...,...,...,...
20845,WA06_001,0.0,0.0,0.0
20846,WA07_001,0.0,0.0,0.0
20847,WA08_001,0.0,0.0,0.0
20848,WA09_001,0.0,0.0,0.0


### Polygon Count by Score Bracket (per spectrum)

Buckets of width 2: `0`, `1-2`, `3-4`, ..., `11-12`, `12+`.

In [44]:
SCORE_BIN_EDGES = [-0.001, 0.5, 2.5, 4.5, 6.5, 8.5, 10.5, 12.5, np.inf]
SCORE_BIN_LABELS = ["0", "1-2", "3-4", "5-6", "7-8", "9-10", "11-12", "12+"]

score_distribution = pd.DataFrame(index=SCORE_BIN_LABELS)
for abbr in SPECTRUM_ABBR.values():
    binned = pd.cut(score_table[f"{abbr}_Score"], bins=SCORE_BIN_EDGES, labels=SCORE_BIN_LABELS)
    score_distribution[abbr] = binned.value_counts().reindex(SCORE_BIN_LABELS)

score_distribution.index.name = "Score"
score_distribution

,L,MR,R
Score,,,
0,15450,14791,18584
1-2,3320,3697,1206
3-4,1818,1700,923
5-6,189,442,125
7-8,43,186,11
9-10,24,32,1
11-12,6,2,0
12+,0,0,0


### Per-Party Disadvantage Flags per Polygon

One row per polygon (`Shape_ID`), 5 questions x 3 spectrum = 15 columns `{question}_{L|MR|R}`, each 0/1.

1. **national_dis** — party is disadvantaged if `df_constituency.Winning_Perspective` for the polygon's constituency is a `Race: X_Y` that doesn't include it (0 if `Head_to_Head` or the party is in the race).
2. **ward_dis** — same rule, but using `df_districts.Winning_Perspective` for the polygon's `(District, District_Ward)`.
3. **char_dis** — from `charisma.json` (`{Left|MR|Right}_Charisma` by `Constituency_Code`): 0 if the party has the highest charisma in its constituency; else 1 if its value is `<4` **and** the leader beats it by `>3`.
4. **Local_dis** — from the Campaign Score section above: 1 if that spectrum's `{L|MR|R}_Score` for the polygon is `<4`.
5. **difference_dis** — from the polygon-level projected vote share (aggregating `electors_long` up from cluster to `Shape_ID`): 1 if any other party leads it by `>=20` points.

In [97]:
PARTIES = ["L", "MR", "R"]
FULL_TO_ABBR = {"Left": "L", "MR": "MR", "R": "R"}

with open(r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\charisma.json") as f:
    charisma = json.load(f)


def canonicalize_head_to_head(value):
    """'Head_to_Head' (df_constituency) and 'Head to Head' (df_districts/index.html) are the
    same neutral value spelled two ways - normalize to one canonical spelling."""
    if isinstance(value, str) and value.strip().replace("_", " ") == "Head to Head":
        return "Head to Head"
    return value


def parse_disadvantage(win_persp):
    """0/0/0 if Head-to-Head/missing/unset.
    'Vote_{P}' (landslide, no second contender) -> P is 0, the other two are 1.
    'Race: {P1}_{P2}' / 'Vote: {P1}_{P2}' (two-party race) -> named parties 0, excluded party 1.
    """
    win_persp = canonicalize_head_to_head(win_persp)
    if not isinstance(win_persp, str) or win_persp.strip() in ("Head to Head", ""):
        return pd.Series({p: 0 for p in PARTIES})

    value = win_persp.strip()

    if value.startswith("Vote_"):
        dominant = FULL_TO_ABBR.get(value[len("Vote_"):], value[len("Vote_"):])
        return pd.Series({p: 0 if p == dominant else 1 for p in PARTIES})

    parties = value.split(":")[-1].strip().split("_")
    in_race = {FULL_TO_ABBR.get(p, p) for p in parties}
    return pd.Series({p: 0 if p in in_race else 1 for p in PARTIES})


# 1. national_dis - via df_constituency.Winning_Perspective
constituency_flags = df_constituency[["Constituency", "Winning_Perspective"]].drop_duplicates(subset=["Constituency"])
constituency_flags = constituency_flags.join(
    constituency_flags["Winning_Perspective"].apply(parse_disadvantage).add_prefix("national_dis_")
)

# 2. ward_dis - via df_districts.Winning_Perspective
ward_flags = df_districts[["District", "District_Ward", "Winning_Perspective"]].drop_duplicates(
    subset=["District", "District_Ward"]
)
ward_flags = ward_flags.join(
    ward_flags["Winning_Perspective"].apply(parse_disadvantage).add_prefix("ward_dis_")
)

disadvantage_table = df_polygon[["Shape_ID", "Constituency", "District", "District_Ward"]].copy()
disadvantage_table = disadvantage_table.merge(
    constituency_flags[["Constituency"] + [f"national_dis_{p}" for p in PARTIES]], on="Constituency", how="left"
)
disadvantage_table = disadvantage_table.merge(
    ward_flags[["District", "District_Ward"] + [f"ward_dis_{p}" for p in PARTIES]],
    on=["District", "District_Ward"],
    how="left",
)

# 3. char_dis - via charisma.json, keyed by Constituency_Code
constituency_code_lookup = sub_region_map[["Constituency", "Constituency_Code"]].drop_duplicates()

charisma_df = pd.DataFrame({"Constituency_Code": list(charisma["Left_Charisma"].keys())})
charisma_df["L"] = charisma_df["Constituency_Code"].map(charisma["Left_Charisma"])
charisma_df["MR"] = charisma_df["Constituency_Code"].map(charisma["MR_Charisma"])
charisma_df["R"] = charisma_df["Constituency_Code"].map(charisma["Right_Charisma"])


def char_dis_row(row):
    """char_dis rule, per party in this row's constituency:
    0 if the party has the highest charisma value (the "leader") of the three.
    Otherwise 1 only if BOTH hold: the party's own value is <4, AND the leader beats it by >3
    (e.g. Left=3, Right leads with 8 -> 8-3=5>3 and 3<4, so char_dis_L=1).
    Otherwise 0 (not leading, but not far enough behind to count as disadvantaged)."""
    vals = row[PARTIES]
    leader = vals.idxmax()
    leader_val = vals[leader]
    return pd.Series(
        {p: (0 if p == leader else int(vals[p] < 4 and (leader_val - vals[p]) > 3)) for p in PARTIES}
    )


char_flags = charisma_df.join(charisma_df.apply(char_dis_row, axis=1).add_prefix("char_dis_"))

disadvantage_table = disadvantage_table.merge(constituency_code_lookup, on="Constituency", how="left")
disadvantage_table = disadvantage_table.merge(
    char_flags[["Constituency_Code"] + [f"char_dis_{p}" for p in PARTIES]], on="Constituency_Code", how="left"
)

# 4. Local_dis - via the Campaign Score section's score_table
local_flags = score_table[["Shape_ID"]].copy()
local_dis_cols = [f"Local_dis_{p}" for p in PARTIES]
for p in PARTIES:
    local_flags[f"Local_dis_{p}"] = (score_table[f"{p}_Score"] < 4).astype(int)

# if every party is flagged in a polygon, it's not meaningfully "disadvantaged" relative to
# the others there (no local campaign activity for anyone) - reset all three to 0
all_flagged = (local_flags[local_dis_cols] == 1).all(axis=1)
local_flags.loc[all_flagged, local_dis_cols] = 0

disadvantage_table = disadvantage_table.merge(local_flags, on="Shape_ID", how="left")

# 5. difference_dis - via projected vote share per polygon (electors_long summed up from cluster to Shape_ID)
polygon_votes = electors_long.groupby("Shape_ID")[["Votes_L", "Votes_MR", "Votes_R", "Valid_Votes"]].sum().reset_index()
for p in PARTIES:
    polygon_votes[f"{p}_pct"] = np.where(
        polygon_votes["Valid_Votes"] > 0, polygon_votes[f"Votes_{p}"] / polygon_votes["Valid_Votes"] * 100, 0.0
    )


def difference_dis_row(row):
    pct = {p: row[f"{p}_pct"] for p in PARTIES}
    return pd.Series({p: int(any((pct[other] - pct[p]) >= 20 for other in PARTIES if other != p)) for p in PARTIES})


diff_flags = polygon_votes[["Shape_ID"]].join(
    polygon_votes.apply(difference_dis_row, axis=1).add_prefix("difference_dis_")
)
disadvantage_table = disadvantage_table.merge(diff_flags, on="Shape_ID", how="left")

question_cols = [f"{q}_{p}" for q in ["national_dis", "ward_dis", "char_dis", "Local_dis", "difference_dis"] for p in PARTIES]

disadvantage_flags = disadvantage_table[["Shape_ID"] + question_cols].copy()
disadvantage_flags[question_cols] = disadvantage_flags[question_cols].fillna(0).astype(int)

disadvantage_flags

,Shape_ID,national_dis_L,national_dis_MR,national_dis_R,ward_dis_L,ward_dis_MR,ward_dis_R,char_dis_L,char_dis_MR,char_dis_R,Local_dis_L,Local_dis_MR,Local_dis_R,difference_dis_L,difference_dis_MR,difference_dis_R
0,A01_001,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1
1,A01_002,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1
2,A01_003,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0
3,A01_004,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1
4,A01_005,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20845,WA06_001,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0
20846,WA07_001,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0
20847,WA08_001,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0
20848,WA09_001,1,0,0,1,0,0,0,0,0,0,0,0,1,1,0


### Disadvantage Flag Pattern Frequency

One row per unique combination of the 15 flags that actually occurs across all polygons, with a `frequency` count and each party's mean projected vote share (`{L|MR|R}_mean_vote`, from the polygon-level `L_pct`/`MR_pct`/`R_pct` computed in the difference_dis step) among the polygons matching that pattern. Columns are reordered so each party's 5 questions sit together (values themselves are unchanged, purely for readability).

In [98]:
questions = ["national_dis", "ward_dis", "char_dis", "Local_dis", "difference_dis"]
party_grouped_cols = [f"{q}_{p}" for p in PARTIES for q in questions]

flags_with_votes = disadvantage_flags.merge(
    polygon_votes[["Shape_ID", "L_pct", "MR_pct", "R_pct"]], on="Shape_ID", how="left"
)

pattern_counts = (
    flags_with_votes.groupby(party_grouped_cols, as_index=False)
    .agg(
        frequency=("Shape_ID", "size"),
        L_mean_vote=("L_pct", "mean"),
        MR_mean_vote=("MR_pct", "mean"),
        R_mean_vote=("R_pct", "mean"),
    )
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)


### Local Perception & Perception Strength

`predict_score_{party} = national_dis_{party} + ward_dis_{party} + difference_dis_{party}` (0-3).

1. **Dominance**: if at least two parties have `predict_score >= 2`, the remaining single party
   with `predict_score < 2` is dominant -> `"{Party}_Dominance"`, strength `0`. (With 3 parties
   this identifies at most one party; if all three are `>=2` there's no dominant party and this
   step is skipped.)
2. Otherwise, a party is **disadvantaged** if: `national_dis==1`, OR (`ward_dis==1` AND `difference_dis==1`). Same rule for all three parties.
3. If more than one party is disadvantaged, prefer whichever have `ward_dis==1`; if none do, fall back to whichever have `national_dis==1`. Exactly one candidate left -> that's *the* disadvantaged party for this pattern; 0 or still >1 -> falls through to Head_to_Head (step 5).
4. For that one disadvantaged party: `Local perception = "Race: {other two parties}"` (e.g. Left disadvantaged -> `"Race: MR_R"`). Strength starts at `predict_score`, then `char_dis`/`Local_dis` each subtract 0.75 (if `predict_score==3`) or 0.5 (if `<=2`) when they're **0** (not 1) - i.e. a redeeming quality softens the disadvantage. Floored, bounded to `[0, 3]`.
5. If no single disadvantaged party was resolved, or the computed strength is `0`, `Local perception = "Head_to_Head"` and strength `0`.

In [ ]:
party_label = {"L": "Left", "MR": "MR", "R": "R"}


def classify_pattern(row):
    predict_score = {p: row[f"national_dis_{p}"] + row[f"ward_dis_{p}"] + row[f"difference_dis_{p}"] for p in PARTIES}

    high_parties = [p for p in PARTIES if predict_score[p] >= 2]
    low_parties = [p for p in PARTIES if predict_score[p] < 2]
    if len(high_parties) >= 2 and len(low_parties) == 1:
        dominant = low_parties[0]
        return pd.Series({"Local perception": f"{party_label[dominant]}_Dominance", "perception strength": 0})

    is_disadvantaged = {
        p: (row[f"national_dis_{p}"] == 1) or (row[f"ward_dis_{p}"] == 1 and row[f"difference_dis_{p}"] == 1)
        for p in PARTIES
    }

    disadvantaged = [p for p in PARTIES if is_disadvantaged[p]]
    if len(disadvantaged) > 1:
        ward_flagged = [p for p in disadvantaged if row[f"ward_dis_{p}"] == 1]
        candidates = ward_flagged if ward_flagged else [p for p in disadvantaged if row[f"national_dis_{p}"] == 1]
    else:
        candidates = disadvantaged

    if len(candidates) == 1:
        party = candidates[0]
        others = [p for p in PARTIES if p != party]
        local_perception = f"Race: {party_label[others[0]]}_{party_label[others[1]]}"

        ps = predict_score[party]
        weight = 0.75 if ps == 3 else 0.5
        char_adj = -weight if row[f"char_dis_{party}"] == 0 else 0
        local_adj = -weight if row[f"Local_dis_{party}"] == 0 else 0
        strength = int(np.floor(ps + char_adj + local_adj))
        strength = max(0, min(3, strength))

        if strength == 0:
            local_perception = "Head_to_Head"
        return pd.Series({"Local perception": local_perception, "perception strength": strength})

    return pd.Series({"Local perception": "Head_to_Head", "perception strength": 0})


# pattern_counts = pattern_counts.join(pattern_counts.apply(classify_pattern, axis=1))

# # pattern_counts.to_csv(
# #     r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\pattern.csv", index=False
# # )

# pattern_counts

,national_dis_L,ward_dis_L,char_dis_L,Local_dis_L,difference_dis_L,national_dis_MR,ward_dis_MR,char_dis_MR,Local_dis_MR,difference_dis_MR,...,ward_dis_R,char_dis_R,Local_dis_R,difference_dis_R,frequency,L_mean_vote,MR_mean_vote,R_mean_vote,Local perception,perception strength
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,2183,42.777828,44.042674,15.608563,Race: Left_MR,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1313,36.263565,38.331705,28.210389,Head_to_Head,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1103,33.796079,36.614639,33.135924,Head_to_Head,0
3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,1,789,44.122765,43.439658,16.453478,Race: Left_MR,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,778,33.894027,36.562018,32.367307,Head_to_Head,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330,1,0,1,1,0,0,0,0,0,0,...,0,1,1,1,1,34.130982,47.984887,20.025189,Race: MR_R,1
331,1,1,0,0,0,0,0,0,0,0,...,0,0,1,0,1,23.371189,43.096234,34.429169,Race: MR_R,1
332,1,1,0,0,0,0,0,0,0,1,...,0,0,0,0,1,27.132701,26.599526,46.771327,Race: MR_R,1
333,1,1,0,0,0,0,0,0,1,0,...,0,0,1,0,1,25.741840,44.807122,30.934718,Race: MR_R,1


### Polygon-level Winning Perspective & Local Perception Embed

`pattern_counts` above validated the disadvantage/classification algorithm (`Local perception`,
`perception strength`), so fold the result back down to polygon level.

Build `df_polygon_embed`: a copy of `df_polygon` with seven columns added -
- `Winning_Perspective_Constituency` - `df_constituency.Winning_Perspective` mapped to each polygon via `Constituency`.
- `Winning_Perspective_Ward` - `df_districts.Winning_Perspective` mapped to each polygon via `(District, District_Ward)`.
- `Local perception` / `perception strength` - resolved per polygon by matching its own 15-flag pattern (`disadvantage_flags`) against `pattern_counts`.
- `L_local_score` / `MR_local_score` / `R_local_score` - the Campaign Score section's `score_table` (`{L|MR|R}_Score`), renamed to avoid confusion with other score columns elsewhere.

The exported CSV keeps only `Shape_ID` and these seven columns.

In [101]:
df_polygon_embed = df_polygon.copy()

# Winning_Perspective at constituency level (national), mapped per polygon
df_polygon_embed = df_polygon_embed.merge(
    df_constituency[["Constituency", "Winning_Perspective"]]
    .drop_duplicates(subset=["Constituency"])
    .rename(columns={"Winning_Perspective": "Winning_Perspective_Constituency"}),
    on="Constituency",
    how="left",
)

# Winning_Perspective at ward level (df_districts), mapped per polygon
df_polygon_embed = df_polygon_embed.merge(
    df_districts[["District", "District_Ward", "Winning_Perspective"]]
    .drop_duplicates(subset=["District", "District_Ward"])
    .rename(columns={"Winning_Perspective": "Winning_Perspective_Ward"}),
    on=["District", "District_Ward"],
    how="left",
)

# Local perception / perception strength, resolved per polygon by matching its 15-flag pattern
# against pattern_counts - via disadvantage_flags, without keeping the 15 flag columns themselves
polygon_local_perception = disadvantage_flags[["Shape_ID"] + party_grouped_cols].merge(
    pattern_counts[party_grouped_cols + ["Local perception", "perception strength"]],
    on=party_grouped_cols,
    how="left",
)[["Shape_ID", "Local perception", "perception strength"]]

df_polygon_embed = df_polygon_embed.merge(polygon_local_perception, on="Shape_ID", how="left")

# Campaign Score section's score_table, renamed L_Score/MR_Score/R_Score -> L_local_score/MR_local_score/R_local_score
df_polygon_embed = df_polygon_embed.merge(
    score_table.rename(columns={f"{p}_Score": f"{p}_local_score" for p in PARTIES}),
    on="Shape_ID",
    how="left",
)

embed_cols = [
    "Shape_ID",
    "Winning_Perspective_Constituency",
    "Winning_Perspective_Ward",
    "Local perception",
    "perception strength",
    "L_local_score",
    "MR_local_score",
    "R_local_score",
]
df_polygon_embed[embed_cols].to_csv(
    r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\df_polygon_embed.csv", index=False
)

df_polygon_embed[embed_cols]

,Shape_ID,Winning_Perspective_Constituency,Winning_Perspective_Ward,Local perception,perception strength,L_local_score,MR_local_score,R_local_score
0,A01_001,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0
1,A01_002,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0
2,A01_003,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0
3,A01_004,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0
4,A01_005,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
20845,WA06_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0
20846,WA07_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0
20847,WA08_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0
20848,WA09_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0


### Turnout Local Factor

Derive `turnout_local_factors` per polygon from `df_polygon_embed`:

1. If `Local perception` is a dominance pattern (`"{Party}_Dominance"`), use it as-is.
2. Else if `Winning_Perspective_Constituency` matches `Local perception` exactly, use that shared
   value - except when both sides are `"Head_to_Head"`, which gets its own label,
   `"Heat Head to Head"`, instead of just repeating `"Head_to_Head"`.
3. Otherwise, `"Normal Head to Head"`.

In [105]:
def turnout_local_factor(row):
    local_perception = row["Local perception"]
    winning_constituency = row["Winning_Perspective_Constituency"]

    if isinstance(local_perception, str) and local_perception.endswith("_Dominance"):
        return local_perception

    if winning_constituency == local_perception:
        if local_perception == "Head_to_Head":
            return "Competitive Head to Head"
        return local_perception

    return "Normal Head to Head"


df_polygon_embed["turnout_local_factors"] = df_polygon_embed.apply(turnout_local_factor, axis=1)

embed_cols = embed_cols + ["turnout_local_factors"]
df_polygon_embed[embed_cols].to_csv(
    r"E:\Coding Site\just_for_fun\Electoral_Modelling\csv_output\df_polygon_embed.csv", index=False
)

df_polygon_embed[embed_cols]

,Shape_ID,Winning_Perspective_Constituency,Winning_Perspective_Ward,Local perception,perception strength,L_local_score,MR_local_score,R_local_score,turnout_local_factors,turnout_local_factors,turnout_local_factors,turnout_local_factors
0,A01_001,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0,Race: Left_MR,Race: Left_MR,Race: Left_MR,Race: Left_MR
1,A01_002,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0,Race: Left_MR,Race: Left_MR,Race: Left_MR,Race: Left_MR
2,A01_003,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0,Race: Left_MR,Race: Left_MR,Race: Left_MR,Race: Left_MR
3,A01_004,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0,Race: Left_MR,Race: Left_MR,Race: Left_MR,Race: Left_MR
4,A01_005,Race: Left_MR,Vote: Left_MR,Race: Left_MR,1,0.0,0.0,0.0,Race: Left_MR,Race: Left_MR,Race: Left_MR,Race: Left_MR
...,...,...,...,...,...,...,...,...,...,...,...,...
20845,WA06_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0,Race: MR_R,Race: MR_R,Race: MR_R,Race: MR_R
20846,WA07_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0,Race: MR_R,Race: MR_R,Race: MR_R,Race: MR_R
20847,WA08_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0,Race: MR_R,Race: MR_R,Race: MR_R,Race: MR_R
20848,WA09_001,Race: MR_R,Vote: MR_R,Race: MR_R,1,0.0,0.0,0.0,Race: MR_R,Race: MR_R,Race: MR_R,Race: MR_R
